In [0]:
%pip install google-api-python-client --quiet
dbutils.library.restartPython()

In [0]:
from googleapiclient.discovery import build

api_key = dbutils.secrets.get(scope="youtube-scope", key="youtube-api-key")

youtube = build("youtube", "v3", developerKey=api_key)

In [0]:
from src.extract import (
    get_youtube_client,
    fetch_channel_data,
    get_uploads_playlist_id,
    fetch_playlist_videos,
    fetch_video_statistics
)

api_key = dbutils.secrets.get(scope="youtube-scope", key="youtube-api-key")

youtube = get_youtube_client(api_key)

channel_id = "UCEGvwrlFT1Ml4VJASkA9rMA"

# Channel
channel_data = fetch_channel_data(youtube, channel_id)

# Playlist
playlist_id = get_uploads_playlist_id(youtube, channel_id)

# Videos
videos = fetch_playlist_videos(youtube, playlist_id)

# Stats
video_ids = [v["snippet"]["resourceId"]["videoId"] for v in videos]
stats = fetch_video_statistics(youtube, video_ids)

In [0]:
from src.transform import (
    json_to_df,
    flatten_channels,
    clean_channels
)

df = json_to_df(spark, channel_data)

df = flatten_channels(df)
df = clean_channels(df)

In [0]:
import importlib
import src.transform
importlib.reload(src.transform)

from src.transform import (
    transform_videos,
    transform_video_stats,
    enrich_videos
)

df_videos = transform_videos(spark, videos)
df_stats = transform_video_stats(spark, stats)

df_final_videos = enrich_videos(df_videos, df_stats)

df_final_videos.display()